# Geneformer Embedding Extraction
**Person 3 - Aleksandar**

Before running: **Runtime > Change runtime type > T4 GPU > Save**

Then: **Runtime > Run all**

In [ ]:
!pip install -q anndata transformers tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')

In [ ]:
# UPDATE THIS to match your shared Google Drive folder name
SHARED_FOLDER_NAME = "shared_folder"

from pathlib import Path
DRIVE_ROOT      = Path(f"/content/drive/MyDrive/{SHARED_FOLDER_NAME}")
INPUT_H5AD      = DRIVE_ROOT / "data/processed/tcga_rna_seq.h5ad"
TOKEN_DICT_PATH = DRIVE_ROOT / "data/processed/token_dictionary.pkl"
OUTPUT_EMB      = DRIVE_ROOT / "data/processed/geneformer_embeddings.npy"
OUTPUT_LABELS   = DRIVE_ROOT / "data/processed/geneformer_labels.npy"

print('Input file exists:', INPUT_H5AD.exists())
print('Token dict exists:', TOKEN_DICT_PATH.exists())

In [ ]:
import anndata as ad
import pickle
import numpy as np
from transformers import AutoModel
from tqdm import tqdm

MODEL_NAME = "ctheodoris/Geneformer"
MAX_LEN    = 2048
BATCH_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Loading AnnData...")
adata = ad.read_h5ad(INPUT_H5AD)
print(f"Samples: {adata.shape[0]}  Genes: {adata.shape[1]}")

print("Loading token dictionary...")
with open(TOKEN_DICT_PATH, "rb") as f:
    token_dict = pickle.load(f)

print("Loading Geneformer from HuggingFace...")
model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)
model.eval().to(device)
print("Model loaded!")

In [ ]:
valid_genes   = [g for g in adata.var_names if g in token_dict]
gene_to_token = {g: token_dict[g] for g in valid_genes}
print(f"Matched {len(valid_genes)}/{adata.shape[1]} genes to Geneformer vocab")

adata  = adata[:, valid_genes].copy()
X      = adata.X
labels = adata.obs["cancer_type"].values

In [ ]:
embeddings = []
print(f"Extracting embeddings for {adata.shape[0]} samples (batch size={BATCH_SIZE})...")

for batch_start in tqdm(range(0, adata.shape[0], BATCH_SIZE)):
    batch_end    = min(batch_start + BATCH_SIZE, adata.shape[0])
    batch_tokens = []

    for i in range(batch_start, batch_end):
        counts = np.array(X[i]).flatten()

        nonzero_mask   = counts > 0
        nonzero_counts = counts[nonzero_mask]
        nonzero_genes  = np.array(valid_genes)[nonzero_mask]
        sort_idx       = np.argsort(-nonzero_counts)
        sorted_genes   = nonzero_genes[sort_idx]

        tokens = [gene_to_token[g] for g in sorted_genes[:MAX_LEN]]
        if len(tokens) < MAX_LEN:
            tokens += [0] * (MAX_LEN - len(tokens))

        batch_tokens.append(tokens)

    input_ids = torch.tensor(batch_tokens).to(device)

    with torch.no_grad():
        outputs        = model(input_ids)
        hidden_states  = outputs.last_hidden_state
        mean_embedding = hidden_states.mean(dim=1)
        embeddings.append(mean_embedding.cpu().numpy())

embeddings = np.concatenate(embeddings, axis=0)
print(f"Done! Embeddings shape: {embeddings.shape}")

In [ ]:
np.save(OUTPUT_EMB,    embeddings)
np.save(OUTPUT_LABELS, labels)

print(f"Embeddings saved to: {OUTPUT_EMB}")
print(f"Labels saved to    : {OUTPUT_LABELS}")
print("\nSamples per cancer type:")
unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {u}: {c}")